In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [65]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler , MinMaxScaler , LabelEncoder,OneHotEncoder,OrdinalEncoder ,Normalizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB , MultinomialNB , BernoulliNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error , accuracy_score , confusion_matrix , classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_selector, make_column_transformer
from sklearn.model_selection import GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

In [42]:
data = pd.read_csv('/content/train.csv')

In [43]:
data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [44]:
data.columns = data.columns.str.lower().str.strip()

In [45]:
X = data.drop(columns=['survived'])
y = data['survived']

In [46]:
data.isnull().sum()

,0
passengerid,0
survived,0
pclass,0
name,0
sex,0
age,177
sibsp,0
parch,0
ticket,0
fare,0


In [64]:
# columnT = ColumnTransformer([
#     ('age_imputer', SimpleImputer(missing_values=np.nan, strategy='mean'), ['age']),
#     ('other_imputer', SimpleImputer(missing_values=np.nan, strategy='most_frequent'), ['pclass', 'sex', 'sibsp', 'parch', 'fare', 'cabin', 'embarked'])
# ],remainder='passthrough')

In [104]:
# SI_mean = SimpleImputer(missing_values=np.nan, strategy='mean')

In [105]:
# SI_mean.fit(X[['age']])
# X['age'] = SI_mean.transform(X[['age']])

# SI_mode = SimpleImputer(missing_values=np.nan, strategy='most_frequent')
# SI_mode.fit(X[['cabin', 'embarked']])
# X[['cabin', 'embarked']] = SI_mode.transform(X[['cabin', 'embarked']])

In [58]:
X.isnull().sum()

,0
pclass,0
sex,0
age,0
sibsp,0
parch,0
fare,0
cabin,0
embarked,0


In [62]:
X = X.drop(columns=['name','ticket','passengerid'])

,pclass,sex,age,sibsp,parch,fare,cabin,embarked
0,3,male,22.000000,1,0,7.2500,B96 B98,S
1,1,female,38.000000,1,0,71.2833,C85,C
2,3,female,26.000000,0,0,7.9250,B96 B98,S
3,1,female,35.000000,1,0,53.1000,C123,S
4,3,male,35.000000,0,0,8.0500,B96 B98,S
...,...,...,...,...,...,...,...,...
886,2,male,27.000000,0,0,13.0000,B96 B98,S
887,1,female,19.000000,0,0,30.0000,B42,S
888,3,female,29.699118,1,2,23.4500,B96 B98,S
889,1,male,26.000000,0,0,30.0000,C148,C


In [63]:
# X_transformed = columnT.fit_transform(X)

# # Get the feature names after transformation
# feature_names = columnT.get_feature_names_out()
# print(feature_names)
# X.head()

In [111]:
from sklearn.pipeline import Pipeline

# Define pipelines for each type of feature transformation
age_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('normalizer', StandardScaler())
])

fare_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

pclass_cabin_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

embarked_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', drop='first'))
])

sex_pipeline = Pipeline([
    ('ohe', OneHotEncoder(handle_unknown='ignore', drop='first'))
])

colmT = ColumnTransformer([
    ('age_pipe', age_pipeline, ['age']),
    ('fare_pipe', fare_pipeline, ['fare']),
    ('pclass_cabin_pipe', pclass_cabin_pipeline, ['pclass', 'cabin']),
    ('embarked_pipe', embarked_pipeline, ['embarked']),
    ('sex_pipe', sex_pipeline, ['sex'])
], remainder='passthrough')

In [112]:
colmT.fit(X)

ColumnTransformer(remainder='passthrough',
                  transformers=[('age_pipe',
                                 Pipeline(steps=[('imputer', SimpleImputer()),
                                                 ('normalizer',
                                                  StandardScaler())]),
                                 ['age']),
                                ('fare_pipe',
                                 Pipeline(steps=[('imputer', SimpleImputer()),
                                                 ('scaler', StandardScaler())]),
                                 ['fare']),
                                ('pclass_cabin_pipe',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('ordinal',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1))]),
                                 ['pclass', 'cabin']),
                                ('embarked_pipe',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('ohe',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'))]),
                                 ['embarked']),
                                ('sex_pipe',
                                 Pipeline(steps=[('ohe',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'))]),
                                 ['sex'])])

In [113]:
X_transformed = colmT.transform(X)
X_transformed_df = pd.DataFrame(X_transformed, columns=colmT.get_feature_names_out())
X_transformed_df.head()

,age_pipe__age,fare_pipe__fare,pclass_cabin_pipe__pclass,pclass_cabin_pipe__cabin,embarked_pipe__embarked_Q,embarked_pipe__embarked_S,sex_pipe__sex_male,remainder__sibsp,remainder__parch
0,-0.592481,-0.502445,2.0,47.0,0.0,1.0,1.0,1.0,0.0
1,0.638789,0.786845,0.0,81.0,0.0,0.0,0.0,1.0,0.0
2,-0.284663,-0.488854,2.0,47.0,0.0,1.0,0.0,0.0,0.0
3,0.407926,0.420730,0.0,55.0,0.0,1.0,0.0,1.0,0.0
4,0.407926,-0.486337,2.0,47.0,0.0,1.0,1.0,0.0,0.0


In [114]:
X_test , X_train , y_test , y_train = train_test_split(X_transformed_df,y,test_size=0.2,random_state=42)

In [115]:
clf1 = LogisticRegression()

In [116]:
clf1.fit(X_train,y_train)

LogisticRegression()

In [118]:
y_pred = clf1.predict(X_test)

In [120]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.82      0.86      0.84       444
           1       0.75      0.68      0.71       268

    accuracy                           0.79       712
   macro avg       0.78      0.77      0.78       712
weighted avg       0.79      0.79      0.79       712



In [121]:
clf2 = DecisionTreeClassifier()

In [122]:
clf2.fit(X_train,y_train)

DecisionTreeClassifier()

In [125]:
y_pred = clf2.predict(X_test)

In [126]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.78      0.78      0.78       444
           1       0.64      0.65      0.64       268

    accuracy                           0.73       712
   macro avg       0.71      0.71      0.71       712
weighted avg       0.73      0.73      0.73       712



In [127]:
clf3 = RandomForestClassifier()

In [128]:
clf3.fit(X_train,y_train)

RandomForestClassifier()

In [130]:
y_pred = clf3.predict(X_test)

In [131]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.83      0.84      0.83       444
           1       0.73      0.72      0.73       268

    accuracy                           0.79       712
   macro avg       0.78      0.78      0.78       712
weighted avg       0.79      0.79      0.79       712



In [132]:
clf4 =KNeighborsClassifier()

In [133]:
clf4.fit(X_train,y_train)

KNeighborsClassifier()

In [134]:
y_pred = clf4.predict(X_test)

In [135]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.79      0.82      0.81       444
           1       0.69      0.65      0.67       268

    accuracy                           0.76       712
   macro avg       0.74      0.73      0.74       712
weighted avg       0.75      0.76      0.75       712



In [137]:
# Create a pipeline connecting the preprocessor (colmT) and clf1 (Logistic Regression)
pipeline_clf1 = Pipeline([
    ('preprocessor', colmT),
    ('classifier', clf1)
])

print('Pipeline for Logistic Regression (clf1) created successfully:')

Pipeline for Logistic Regression (clf1) created successfully:


In [138]:
df = pd.read_csv('/content/test.csv')

In [141]:
df

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...
413,1305,3,"Spector, Mr. Woolf",male,NaN,0,0,A.5. 3236,8.0500,NaN,S
414,1306,1,"Oliva y Ocana, Dona. Fermina",female,39.0,0,0,PC 17758,108.9000,C105,C
415,1307,3,"Saether, Mr. Simon Sivertsen",male,38.5,0,0,SOTON/O.Q. 3101262,7.2500,NaN,S
416,1308,3,"Ware, Mr. Frederick",male,NaN,0,0,359309,8.0500,NaN,S


In [145]:
df.columns = df.columns.str.lower().str.strip()
df.drop(columns=['name','ticket','passengerid'],inplace=True )

In [146]:
X_test_transformed = colmT.transform(df)
X_test_transformed_df = pd.DataFrame(X_test_transformed, columns=colmT.get_feature_names_out())

In [147]:
test_predictions = pipeline_clf1.predict(df)
print("Predictions for the test data using Logistic Regression:\n", test_predictions)

Predictions for the test data using Logistic Regression:
 [0 0 0 0 0 0 1 0 1 0 0 0 1 0 1 1 0 0 0 0 0 0 1 1 1 0 1 0 0 0 0 0 0 0 1 0 0
 1 0 0 0 0 0 1 1 0 0 0 1 0 1 0 1 1 0 0 0 0 0 1 0 0 0 1 1 1 1 0 1 1 1 0 0 1
 1 1 0 1 0 1 0 0 0 0 0 0 1 1 1 1 0 0 1 0 1 0 0 0 1 0 1 0 0 0 1 0 0 0 0 0 0
 1 1 1 1 0 0 1 1 1 1 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 0 0 0 1 0
 1 0 1 0 0 0 0 0 1 1 0 1 1 0 1 0 0 0 0 0 1 1 0 0 0 0 0 1 1 0 1 1 0 0 1 0 1
 0 1 0 0 0 0 1 0 0 1 0 1 1 0 0 1 0 0 1 0 1 1 0 1 0 0 0 0 0 0 0 1 0 1 0 1 0
 1 0 1 1 0 1 0 0 0 1 0 0 1 0 0 0 1 1 1 1 1 0 0 0 1 0 1 0 1 0 1 0 0 0 0 0 1
 0 0 0 1 1 0 0 1 0 0 0 0 0 1 1 0 1 0 0 0 0 1 0 1 1 1 0 0 1 0 0 1 1 0 0 0 0
 1 0 1 0 0 0 0 0 1 0 1 0 0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 1 1 0 1 0 0 0 1 0 0
 1 0 1 0 0 0 0 0 0 0 1 0 1 0 0 0 1 1 0 0 0 1 0 1 0 0 0 0 1 1 0 1 1 0 1 1 0
 0 1 0 0 1 1 0 0 0 0 0 0 0 1 0 1 0 0 0 0 1 1 0 0 0 1 0 1 0 0 1 0 1 1 0 0 0
 0 1 1 1 1 0 0 1 0 0 0]
